hw7 
==========

In [2]:
import os
import time
from pathlib import Path
from collections import Counter

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf



from sklearn.model_selection import train_test_split

from tensorflow.keras import Input, Sequential
from tensorflow.keras.layers import (
    Embedding,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reproducibility
random_seed = 42
tf.keras.utils.set_random_seed(random_seed)

# Utility
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))


print("Visible GPUs:", tf.config.get_visible_devices("GPU"))

Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


---
# premade functions

In [3]:
def plot_learning_curves(history, title="Learning Curves", show=True):

    h = history.history

    best_idx = int(np.argmin(h["val_loss"]))
    best_epoch = best_idx + 1
    epochs = np.arange(1, len(h["loss"]) + 1)

    fig, axs = plt.subplots(
        2, 1, figsize=(8, 7), sharex=True
    )

    # Loss
    axs[0].plot(epochs, h["loss"], label="Training")
    axs[0].plot(epochs, h["val_loss"], label="Validation")
    axs[0].scatter(
        best_epoch, h["val_loss"][best_idx],
        color="red", marker="x", s=70,
        label=f"Best epoch: {best_epoch}"
    )
    axs[0].set_title(f"{title} — Binary Cross-Entropy")
    axs[0].set_ylabel("Loss")
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)

    # Accuracy
    axs[1].plot(epochs, h["accuracy"], label="Training")
    axs[1].plot(epochs, h["val_accuracy"], label="Validation")
    axs[1].scatter(
        best_epoch, h["val_accuracy"][best_idx],
        color="red", marker="x", s=70,
        label="Accuracy at best epoch"
    )
    axs[1].set_title("Accuracy")
    axs[1].set_xlabel("Epoch")
    axs[1].set_ylabel("Accuracy")
    axs[1].set_ylim(0, 1.05)
    axs[1].legend()
    axs[1].grid(True, alpha=0.3)

    plt.tight_layout()

    if show:
        plt.show()
    else:
        plt.close(fig)

    return best_idx


In [4]:
def train_and_validate(
    model,
    X_train, y_train,
    X_val, y_val,
    title="Learning Curves",
    epochs=200,
    optimizer="Adam",
    lr_schedule=1e-3,
    clipnorm=None,
    batch_size=128,
    loss="binary_crossentropy",
    metrics=None,
    use_early_stopping=True,
    patience=10,
    min_delta=0.0001,
    callbacks=None,
    verbose=0,
    plot=True
):

    if metrics is None:
        metrics = ["accuracy"]

    if callbacks is None:
        callbacks = []

    print(f"\n{title}\n")

    if optimizer == "Adam":
        opt = Adam(
            learning_rate=lr_schedule,
            clipnorm=clipnorm
        )
    else:
        opt = optimizer

    model.compile(
        optimizer=opt,
        loss=loss,
        metrics=metrics
    )

    cbs = list(callbacks)

    if use_early_stopping:
        cbs.insert(0, EarlyStopping(
            monitor="val_loss",
            patience=patience,
            min_delta=min_delta,
            restore_best_weights=True,
            verbose=verbose
        ))

    start = time.time()

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=cbs,
        verbose=verbose
    )

    elapsed = time.time() - start

    h = history.history
    best_idx = int(np.argmin(h["val_loss"]))
    best_epoch = best_idx + 1
    num_epochs = len(h["loss"])

    best_val_loss = float(h["val_loss"][best_idx])
    best_val_acc = float(h["val_accuracy"][best_idx])
    train_acc_at_best = float(h["accuracy"][best_idx])

    if plot:
        plot_learning_curves(history, title=title)

    print(f"Epochs Trained:                {num_epochs}")
    print(f"Best Epoch:                    {best_epoch}")
    print(f"Training Accuracy @ Best:      {train_acc_at_best:.4f}")
    print(f"Validation Loss @ Best:        {best_val_loss:.4f}")
    print(f"Validation Accuracy @ Best:    {best_val_acc:.4f}")
    print(f"Execution Time:                {format_hms(elapsed)}")

    return {
            "title": title,
            "best_epoch": best_epoch,
            "epochs_trained": num_epochs,
            "Val Loss @ Best": best_val_loss,
            "Val Acc @ Best": best_val_acc,
            "Execution Seconds": elapsed
        }


In [5]:
def prepare_sequences(raw_sequences, vocabulary_size, max_length):
    """Apply vocabulary cap, then pad/truncate sequences."""

    capped = [
        [token if token < vocabulary_size else 2
         for token in review]
        for review in raw_sequences
    ]

    return pad_sequences(
        capped,
        maxlen=max_length,
        padding="post",
        truncating="post"
    )

---

# p0

## dataset

In [6]:
training_vocabulary_size = 50_000       # <-----
max_text_length          = 500          # <-----
DATA_FRACTION            = 0.25         # <----- 1.0 for final submission

assert 0 < DATA_FRACTION <= 1.0

In [7]:

# ------------------------------------------------------------
# 1. Load all reviews without restricting the vocabulary
# ------------------------------------------------------------

(X_tr, y_tr), (X_te, y_te) = imdb.load_data()

X = np.concatenate([X_tr, X_te], axis=0)
y = np.concatenate([y_tr, y_te], axis=0)

# ------------------------------------------------------------
# 2. Stratified 80/10/10 train/validation/test split
# ------------------------------------------------------------

X_train_raw, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=random_seed,
    stratify=y
)

X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=random_seed,
    stratify=y_temp
)

# ------------------------------------------------------------
# 3. Training-data throttle
#    Keep validation and test sets unchanged
# ------------------------------------------------------------

if DATA_FRACTION < 1.0:
    X_train_raw, _, y_train, _ = train_test_split(
        X_train_raw,
        y_train,
        train_size=DATA_FRACTION,
        random_state=random_seed,
        stratify=y_train
    )

print(f"DATA_FRACTION:      {DATA_FRACTION}")
print(f"Training reviews:   {len(X_train_raw):,}")
print(f"Validation reviews: {len(X_val_raw):,}")
print(f"Test reviews:       {len(X_test_raw):,}")

# ------------------------------------------------------------
# 4. Apply vocabulary restriction and pad/truncate sequences
#    Preserve the raw sequences for later experiments
# ------------------------------------------------------------

X_train = prepare_sequences(
    X_train_raw, training_vocabulary_size, max_text_length
)

X_val = prepare_sequences(
    X_val_raw, training_vocabulary_size, max_text_length
)

X_test = prepare_sequences(
    X_test_raw, training_vocabulary_size, max_text_length
)

print()
print("Training:  ", X_train.shape)
print("Validation:", X_val.shape)
print("Test:      ", X_test.shape)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
DATA_FRACTION:      0.25
Training reviews:   10,000
Validation reviews: 5,000
Test reviews:       5,000

Training:   (10000, 500)
Validation: (5000, 500)
Test:       (5000, 500)


## review

In [ ]:

# # ------------------------------------------------------------
# # 1. Global vocabulary statistics
# # ------------------------------------------------------------

# word_index = imdb.get_word_index()
# total_unique_words_corpus = len(word_index)

# print(f"Total unique words in Keras IMDB word_index "
#       f"(corpus-wide): {total_unique_words_corpus:,}")

# # ------------------------------------------------------------
# # 2. Review length statistics
# # ------------------------------------------------------------

# lengths = np.fromiter(
#     (len(review) for review in X_train_raw),
#     dtype=np.int32
# )

# print(f"\nNumber of training reviews: {len(lengths):,}")
# print(f"Max length: {lengths.max():,} tokens")
# print(f"Mean length: {lengths.mean():.1f} tokens")
# print(f"Median length: {np.median(lengths):.0f} tokens")

# for q in (90, 95, 99):
#     print(f"{q}th percentile: "
#           f"{int(np.percentile(lengths, q))} tokens")

# plt.figure(figsize=(8, 5))
# plt.hist(lengths, bins=100, edgecolor="blue")
# plt.title("Histogram of IMDB Review Lengths (Training Set)")
# plt.xlabel("Tokens per review")
# plt.ylabel("Frequency")
# plt.grid(axis="y", alpha=0.3)
# plt.show()


In [ ]:

# # ------------------------------------------------------------
# # 3. Token frequency counts (training data only)
# #
# # IMDB token IDs:
# #   0 = PAD
# #   1 = START
# #   2 = OOV
# #   3 = UNUSED
# #   4+ = actual words
# # ------------------------------------------------------------

# train_counts = Counter(
#     token
#     for review in X_train_raw
#     for token in review
#     if token >= 4
# )

# total_tokens_train_words = sum(train_counts.values())
# num_unique_words_train = len(train_counts)

# print(f"\nUnique word types in TRAIN: "
#       f"{num_unique_words_train:,}")
# print(f"Total word tokens in TRAIN: "
#       f"{total_tokens_train_words:,}")

# # ------------------------------------------------------------
# # 4. Calculate vocabulary coverage
# #
# # Keras assigns lower token IDs to more frequent words
# # in the original IMDB vocabulary.
# #
# # A vocabulary cap K retains word IDs below K.
# # All remaining word IDs become OOV.
# # ------------------------------------------------------------

# def coverage_for_cap(num_words_cap):

#     kept_types = sum(
#         token < num_words_cap
#         for token in train_counts
#     )

#     kept_tokens = sum(
#         count
#         for token, count in train_counts.items()
#         if token < num_words_cap
#     )

#     oov_tokens = total_tokens_train_words - kept_tokens

#     coverage = kept_tokens / total_tokens_train_words
#     oov_share = oov_tokens / total_tokens_train_words

#     return {
#         "cap": num_words_cap,
#         "kept_tokens": kept_tokens,
#         "kept_types": kept_types,
#         "coverage": coverage,
#         "oov_share": oov_share
#     }

# # ------------------------------------------------------------
# # 5. Vocabulary coverage table
# # ------------------------------------------------------------

# candidate_caps = [
#     2_000, 5_000, 10_000, 20_000,
#     30_000, 50_000, 70_000, 88_588
# ]

# rows = [coverage_for_cap(k) for k in candidate_caps]

# print("\nCoverage metrics by candidate vocabulary size "
#       "(TRAIN split):")

# for r in rows:
#     print(
#         f"training_vocabulary_size={r['cap']:>7,} | "
#         f"coverage={r['coverage']*100:6.2f}% | "
#         f"OOV={r['oov_share']*100:6.2f}% | "
#         f"types kept={r['kept_types']:>7,} | "
#         f"tokens kept={r['kept_tokens']:>10,}"
#     )

# # ------------------------------------------------------------
# # 6. Vocabulary coverage plot
# # ------------------------------------------------------------

# plt.figure(figsize=(8, 5))

# plt.plot(
#     candidate_caps,
#     [r["coverage"] for r in rows],
#     marker="o"
# )

# plt.xlabel("training_vocabulary_size (num_words)")
# plt.ylabel("Token coverage on TRAIN (fraction)")
# plt.title("Token Coverage vs. Vocabulary Size (TRAIN only)")
# plt.grid(True, linestyle=":", alpha=0.6)
# plt.ylim(0.75, 1.02)
# plt.show()


    ---

## GloVe

In [8]:
embedding_dimension = 100                 # <-----

word_index = imdb.get_word_index()

# GloVe 6B provides 50, 100, 200, and 300 dimensions
assert embedding_dimension in (50, 100, 200, 300)

glove_path = Path(
    f"../glove.6B/glove.6B.{embedding_dimension}d.txt"
)

if not glove_path.exists():
    raise FileNotFoundError(f"GloVe file not found: {glove_path}")

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 8s 5us/step


In [9]:
embeddings_index = {}

with glove_path.open(encoding="utf8") as f:
    for line in f:
        word, *vec = line.split()
        embeddings_index[word] = np.asarray(
            vec, dtype="float32"
        )

print(f"Loaded {len(embeddings_index):,} GloVe vectors "
      f"of dimension {embedding_dimension}")

# ------------------------------------------------------------
# Construct embedding matrix
# ------------------------------------------------------------

embedding_matrix = np.zeros(
    (training_vocabulary_size, embedding_dimension),
    dtype="float32"
)

matched = 0
total_words = 0

for word, raw_idx in word_index.items():

    idx = raw_idx + 3  # Keras IMDB reserved-token offset

    if idx < training_vocabulary_size:
        total_words += 1

        vec = embeddings_index.get(word)

        if vec is not None:
            embedding_matrix[idx] = vec
            matched += 1

# ------------------------------------------------------------
# Report embedding coverage
# ------------------------------------------------------------

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print(f"Vocabulary words:       {total_words:,}")
print(f"Words found in GloVe:   {matched:,}")
print(f"Words missing in GloVe: {total_words - matched:,}")
print(f"GloVe word coverage:    {matched / total_words:.2%}")

Loaded 400,000 GloVe vectors of dimension 100
Embedding matrix shape: (50000, 100)
Vocabulary words:       49,996
Words found in GloVe:   42,330
Words missing in GloVe: 7,666
GloVe word coverage:    84.67%


## base model

In [ ]:

# # ==================================================
# # Baseline: Frozen GloVe Embeddings
# # ==================================================

# model_baseline_frozen = Sequential([
#     Input(shape=(max_text_length,), dtype="int32"),

#     Embedding(
#         input_dim=training_vocabulary_size,
#         output_dim=embedding_dimension,
#         weights=[embedding_matrix],
#         mask_zero=True,          # Exclude padding from pooling
#         trainable=False          # Freeze pretrained embeddings
#     ),

#     GlobalAveragePooling1D(),    # Average word embeddings
#     Dense(1, activation="sigmoid")
# ])

# frozen_results = train_and_validate(
#     model_baseline_frozen,
#     X_train, y_train,
#     X_val, y_val,
#     epochs = 100, 
#     lr_schedule=1e-2,
#     title="Baseline — Frozen GloVe"
# )
